In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
df = pd.read_csv(
    "../data/processed/feature_engineered_sales_data.csv"
)
df

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,...,Month,Day,DayOfWeek,Quarter,Price_Difference,Price_Ratio,Price_Difference_Percent,Effective_Price,Stock_Status,Inventory_to_Order_Ratio
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,...,1,1,5,1,-13.01,0.848244,-15.175551,69.084,In_Stock,0.770751
1,2022-01-01,S004,P0013,Groceries,West,136,104,385,20.24,10,...,1,1,5,1,1.34,1.070899,7.089947,18.216,In_Stock,0.352332
2,2022-01-01,S004,P0012,Electronics,West,111,111,113,118.15,0,...,1,1,5,1,-15.31,0.885284,-11.471602,118.150,In_Stock,0.973684
3,2022-01-01,S004,P0011,Clothing,West,195,60,293,52.89,0,...,1,1,5,1,-9.40,0.849093,-15.090705,52.890,In_Stock,0.663265
4,2022-01-01,S004,P0010,Groceries,West,223,120,597,30.02,0,...,1,1,5,1,0.87,1.029846,2.984563,30.020,In_Stock,0.372910
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75995,2024-01-30,S002,P0008,Groceries,South,85,81,94,26.98,0,...,1,30,1,1,2.33,1.094523,9.452333,26.980,In_Stock,0.894737
75996,2024-01-30,S002,P0007,Toys,South,414,126,0,26.84,10,...,1,30,1,1,-2.71,0.908291,-9.170897,24.156,In_Stock,414.000000
75997,2024-01-30,S002,P0006,Clothing,South,98,96,117,63.58,10,...,1,30,1,1,-12.64,0.834164,-16.583574,57.222,In_Stock,0.830508
75998,2024-01-30,S002,P0016,Clothing,South,88,78,0,91.50,5,...,1,30,1,1,5.76,1.067180,6.717985,86.925,In_Stock,88.000000


In [4]:
df.shape

(76000, 27)

In [5]:
y = df["Demand"]

In [6]:
feature_columns = [
    "Category",
    "Region",
    "Inventory Level",
    "Price",
    "Discount",
    "Weather Condition",
    "Promotion",
    "Competitor Pricing",
    "Seasonality",
    "Epidemic",
    "Year",
    "Month",
    "Day",
    "DayOfWeek",
    "Price_Difference",
    "Price_Difference_Percent",
    "Stock_Status",
    "Inventory_to_Order_Ratio"
]

X = df[feature_columns].copy()

In [7]:
split_index = int(len(X) * 0.80)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

In [8]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (60800, 18)
X_test : (15200, 18)
y_train: (60800,)
y_test : (15200,)


In [9]:
categorical_features = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print(categorical_features)

['Category', 'Region', 'Weather Condition', 'Seasonality', 'Stock_Status']


In [10]:
numerical_features = X_train.select_dtypes(
    include=[np.number]
).columns.tolist()

print(numerical_features)

['Inventory Level', 'Price', 'Discount', 'Promotion', 'Competitor Pricing', 'Epidemic', 'Year', 'Month', 'Day', 'DayOfWeek', 'Price_Difference', 'Price_Difference_Percent', 'Inventory_to_Order_Ratio']


In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        )
    ]
)

In [12]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [13]:
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"]
}

In [14]:
random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_grid,
    n_iter=10,
    cv=3,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [15]:
random_search.fit(
    X_train,
    y_train
)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__max_depth': [None, 10, ...], 'model__max_features': ['sqrt', 'log2'], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estim

In [16]:
print("Best Parameters:")
print(random_search.best_params_)

Best Parameters:
{'model__n_estimators': 100, 'model__min_samples_split': 5, 'model__min_samples_leaf': 1, 'model__max_features': 'sqrt', 'model__max_depth': 20}


In [17]:
tuned_model = random_search.best_estimator_

In [18]:
y_pred_tuned = tuned_model.predict(X_test)

In [19]:
mae_tuned = mean_absolute_error(
    y_test,
    y_pred_tuned
)

print("MAE:", mae_tuned)

MAE: 19.828231735124007


In [20]:
rmse_tuned = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_tuned
    )
)
print("RMSE:", rmse_tuned)

RMSE: 25.88058369955579


In [21]:
r2_tuned = r2_score(
    y_test,
    y_pred_tuned
)

print("R²:", r2_tuned)

R²: 0.6553833624088288


In [22]:
print("Tuned Random Forest")
print("-------------------")
print("MAE :", mae_tuned)
print("RMSE:", rmse_tuned)
print("R²  :", r2_tuned)

Tuned Random Forest
-------------------
MAE : 19.828231735124007
RMSE: 25.88058369955579
R²  : 0.6553833624088288


In [23]:
comparison = pd.DataFrame({
    "Model": [
        "Original Random Forest",
        "Tuned Random Forest"
    ],
    "MAE": [
        17.639030,
        mae_tuned
    ],
    "RMSE": [
        24.078557,
        rmse_tuned
    ],
    "R2": [
        0.701703,
        r2_tuned
    ]
})

comparison

,Model,MAE,RMSE,R2
0,Original Random Forest,17.639030,24.078557,0.701703
1,Tuned Random Forest,19.828232,25.880584,0.655383


In [24]:
if r2_tuned > 0.701703:
    print("Tuned model improved the R² score.")

elif r2_tuned < 0.701703:
    print("Original model performed better.")

else:
    print("Both models have the same R² score.")

Original model performed better.


In [25]:
final_model = tuned_model

print("Final model selected.")

Final model selected.


In [26]:
comparison.to_csv(
    "../data/processed/model_tuning_results.csv",
    index=False
)

In [27]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(
    final_model,
    "../models/final_demand_model.pkl"
)

print("Tuned model saved successfully.")

Tuned model saved successfully.
